# 🔥 Fire Detection YOLOv9 Training - Google Colab
## Panduan Lengkap Training Model Deteksi Api dengan YOLOv9 di Google Colab (GRATIS)

### 📌 Persyaratan:
- Akun Google (untuk Colab dan Drive)
- Dataset sudah diupload ke Google Drive
- Koneksi internet stabil

### ⏱️ Estimasi Waktu:
- Install library: 1-2 menit
- Training 50 epoch: **2-4 jam** (dengan GPU Colab)
- Validation: 5-10 menit
- Total: **2.5-5 jam**

## 📦 STEP 1: Install Ultralytics YOLOv9

In [ ]:
# Install Ultralytics (YOLOv9)
!pip install ultralytics

# Cek versi
import ultralytics
print(f"✅ Ultralytics version: {ultralytics.__version__}")

# Cek CUDA (GPU)
import torch
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  WARNING: GPU tidak tersedia, pakai CPU")

## 💾 STEP 2: Mount Google Drive
Upload dataset ke Google Drive terlebih dahulu, lalu mount di sini.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# List isi drive untuk cek
!ls "/content/drive/MyDrive/"

## 📂 STEP 3: Set Path Dataset
**GANTI PATH INI sesuai lokasi dataset Anda di Google Drive!**

Contoh struktur yang benar:
```
/content/drive/MyDrive/YOLOv9_Fire_Detection/FIRE DETECTION USING YOLOV9/DATASET/data.yaml
```

In [ ]:
import os

# ===== GANTI PATH INI =====
DATASET_PATH = "/content/drive/MyDrive/YOLOv9_Fire_Detection/FIRE DETECTION USING YOLOV9/DATASET/data.yaml"

# Cek apakah dataset ada
if os.path.exists(DATASET_PATH):
    print(f"✅ Dataset ditemukan di: {DATASET_PATH}")
    
    # Cek struktur dataset
    dataset_dir = os.path.dirname(DATASET_PATH)
    print(f"\n📁 Struktur dataset:")
    for root, dirs, files in os.walk(dataset_dir):
        level = root.replace(dataset_dir, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 2 * (level + 1)
        for file in files[:5]:  # Tampilkan 5 file pertama
            print(f'{subindent}{file}')
        if len(files) > 5:
            print(f'{subindent}... dan {len(files) - 5} file lainnya')
else:
    print(f"❌ Dataset TIDAK ditemukan di: {DATASET_PATH}")
    print("\n🔧 Solusi:")
    print("1. Pastikan dataset sudah diupload ke Google Drive")
    print("2. Cek path-nya benar")
    print("3. Gunakan !ls untuk lihat isi drive")


## 🚀 STEP 4: Training Model
### Konfigurasi Training:
- Model: yolov9c.pt (pretrained)
- Epochs: 50
- Image size: 640px
- Batch size: 16 (optimal untuk GPU Colab)
- Device: GPU (T4 atau A100)

**Estimasi waktu: 2-4 jam**

In [ ]:
from ultralytics import YOLO
import torch

# Gunakan GPU jika tersedia
device = "0" if torch.cuda.is_available() else "cpu"
print(f"🔥 Training dengan: {'GPU' if device == '0' else 'CPU'}")

# Load model
print("📦 Loading pretrained model yolov9c.pt...")
model = YOLO("yolov9c.pt")

# Start training
print("🚀 Starting training...")
print("⏳ Training akan memakan waktu 2-4 jam")
print("💡 Tekan 'Stop' di menu Runtime jika mau menghentikan")

results = model.train(
    data=DATASET_PATH,
    epochs=50,
    imgsz=640,
    batch=16,
    name="fire_detection_yolov9_colab",
    patience=20,
    device=device,
    workers=2,
    verbose=True,
    plots=True,
)

print("\n" + "="*60)
print("✅ TRAINING SELESAI!")
print("="*60)
print(f"📁 Model tersimpan di: /content/runs/detect/fire_detection_yolov9_colab/")
print(f"🏆 Best model: /content/runs/detect/fire_detection_yolov9_colab/weights/best.pt")

## 🔍 STEP 5: Validation Model
Validasi model yang sudah di-train untuk lihat performa.

In [ ]:
print("🔍 Validating model...")

# Load best model
model_path = "/content/runs/detect/fire_detection_yolov9_colab/weights/best.pt"
if os.path.exists(model_path):
    model = YOLO(model_path)
    
    # Run validation
    results = model.val(
        data=DATASET_PATH,
        imgsz=640,
        batch=16,
        device=device,
    )
    
    print("\n" + "="*60)
    print("✅ VALIDATION SELESAI!")
    print("="*60)
    print(f"📊 Metrics:")
    print(f"   - mAP50: {results.box.map50:.4f}")
    print(f"   - mAP50-95: {results.box.map:.4f}")
    print(f"   - Precision: {results.box.p:.4f}")
    print(f"   - Recall: {results.box.r:.4f}")
    
    # Target: mAP50 > 0.7 (70%)
    if results.box.map50 > 0.7:
        print("\n🎉 EXCELLENT! Model sangat bagus!")
    elif results.box.map50 > 0.5:
        print("\n👍 GOOD! Model cukup bagus")
    else:
        print("\n⚠️  Model perlu training lebih lama")
else:
    print(f"❌ Model tidak ditemukan di: {model_path}")
    print("Jalankan training terlebih dahulu!")

## 📤 STEP 6: Export Model (Optional)
Export model ke format lain untuk deployment.

In [ ]:
print("📤 Exporting model...")

# Export ke ONNX (untuk CPU inference)
print("\n🔄 Exporting to ONNX...")
model.export(format="onnx")
print("✅ ONNX model tersimpan di: /content/runs/detect/fire_detection_yolov9_colab/weights/best.onnx")

# Export ke TFLite (untuk mobile/edge devices)
print("\n🔄 Exporting to TFLite...")
model.export(format="tflite")
print("✅ TFLite model tersimpan di: /content/runs/detect/fire_detection_yolov9_colab/weights/best.tflite")

# Export ke TensorRT (untuk NVIDIA GPU)
# print("\n🔄 Exporting to TensorRT...")
# model.export(format="engine")  # Butuh TensorRT installed

print("\n✅ Semua export selesai!")

## 💾 STEP 7: Download Hasil ke Komputer
Download model yang sudah di-train ke komputer Anda.

In [ ]:
from google.colab import files
import glob

print("💾 Downloading hasil training...")

# Download best.pt
best_pt_path = "/content/runs/detect/fire_detection_yolov9_colab/weights/best.pt"
if os.path.exists(best_pt_path):
    files.download(best_pt_path)
    print(f"✅ {os.path.basename(best_pt_path)} sudah didownload")

# Download best.onnx
best_onnx_path = "/content/runs/detect/fire_detection_yolov9_colab/weights/best.onnx"
if os.path.exists(best_onnx_path):
    files.download(best_onnx_path)
    print(f"✅ {os.path.basename(best_onnx_path)} sudah didownload")

# Download best.tflite
best_tflite_path = "/content/runs/detect/fire_detection_yolov9_colab/weights/best.tflite"
if os.path.exists(best_tflite_path):
    files.download(best_tflite_path)
    print(f"✅ {os.path.basename(best_tflite_path)} sudah didownload")

# Download results.txt (metrics)
results_txt = glob.glob("/content/runs/detect/fire_detection_yolov9_colab/results*.txt")
if results_txt:
    files.download(results_txt[0])
    print(f"✅ {os.path.basename(results_txt[0])} sudah didownload")

print("\n🎉 Semua file sudah didownload!")
print("📁 Simpan di folder proyek Anda: d:\\A3-HXCODE\\")

## 🎯 STEP 8: Testing Model (Optional)
Test model dengan gambar custom untuk lihat hasil deteksi.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# Load model
model = YOLO("/content/runs/detect/fire_detection_yolov9_colab/weights/best.pt")

# Upload gambar test
from google.colab import files
uploaded = files.upload()

if uploaded:
    # Ambil gambar pertama yang diupload
    test_image_path = list(uploaded.keys())[0]
    
    # Predict
    results = model.predict(test_image_path, conf=0.5)
    
    # Tampilkan gambar dengan bounding box
    img = Image.open(test_image_path)
    img_array = np.array(img)
    
    # Plot
    plt.figure(figsize=(10, 10))
    plt.imshow(img_array)
    
    # Draw bounding boxes
    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            cls = int(box.cls[0])
            
            # Draw rectangle
            plt.gca().add_patch(plt.Rectangle(
                (x1, y1), x2-x1, y2-y1, 
                fill=False, edgecolor='red', linewidth=3
            ))
            # Add label
            plt.text(x1, y1, f'Fire {conf:.2f}', 
                    color='red', fontsize=12, 
                    bbox=dict(facecolor='yellow', alpha=0.5))
    
    plt.axis('off')
    plt.title('Fire Detection Result')
    plt.show()
    
    # Print results
    print(f"\n🔍 Hasil deteksi:")
    print(f"   - Jumlah objek terdeteksi: {len(results[0].boxes)}")
    for i, box in enumerate(results[0].boxes):
        print(f"   - Objek {i+1}: Fire (confidence: {float(box.conf[0]):.2f})")
else:
    print("❌ Tidak ada gambar diupload")

## 📚 Tips & Troubleshooting

### ⚡ Tips untuk Training Lebih Baik:

1. **Jika GPU tidak tersedia:**
   - Cek apakah Colab memberikan GPU (Runtime > Change runtime type > GPU)
   - Jika tidak ada GPU, coba refresh halaman atau buka notebook baru

2. **Jika training lambat:**
   - Kurangi batch size dari 16 ke 8
   - Kurangi epochs dari 50 ke 30

3. **Jika memory habis:**
   - Kurangi batch size
   - Kurangi image size dari 640 ke 384

4. **Jika dataset tidak ditemukan:**
   - Pastikan path benar
   - Cek struktur folder di Google Drive

### 🎯 Target Metrics:
- **mAP50 > 0.7** = Model sangat bagus
- **mAP50 > 0.5** = Model cukup bagus
- **mAP50 < 0.3** = Model perlu training lebih lama

### ⏱️ Estimasi Waktu:
- 1 epoch: ~3-5 menit
- 10 epoch: ~30-50 menit
- 50 epoch: ~2.5-4 jam

### 💾 Penyimpanan:
- Model tersimpan di: `/content/runs/detect/fire_detection_yolov9_colab/`
- Download sebelum menutup notebook (file hilang jika session berakhir)

### 📋 Ringkasan:
1. ✅ Install Ultralytics
2. ✅ Mount Google Drive
3. ✅ Set path dataset
4. ✅ Training model (2-4 jam)
5. ✅ Validation model
6. ✅ Export model (ONNX, TFLite)
7. ✅ Download hasil

### 🔗 Link Penting:
- [Ultralytics YOLOv9 Docs](https://docs.ultralytics.com/)
- [Google Colab](https://colab.research.google.com/)
- [Roboflow Dataset](https://universe.roboflow.com/)